# 03. Data Preparation: Agregasi Centroid Poligon & Stratified Split
## Mitigasi Kebocoran Data Spasial (*Spatial Data Leakage*)

Piksel-piksel yang berada di dalam satu poligon sampel memiliki autokorelasi spasial (*spatial autocorrelation*) yang sangat tinggi. Jika data dibagi (*split*) acak pada level piksel, model akan menghafal koordinat/lokasi poligon (akurasi semu $>99\%$). 
Oleh karena itu, pada notebook ini dilakukan:
1. **Agregasi Centroid per `poligon_id`**: Menghitung nilai rata-rata dari 9 fitur spektral untuk setiap poligon unik.
2. **Stratified Train-Test Split (80:20)**: Membagi 357 poligon menjadi 285 poligon latih dan 72 poligon uji dengan proporsi seimbang untuk 6 kelas (`random_state=42`).
3. Menyimpan hasil pembagian ke `output/df_poligon_split.csv`.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = DATA_DIR / "piksel_s2.csv"
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"File {DATA_FILE} tidak ditemukan! Pastikan file 'piksel_s2.csv' tersedia di folder data/."
    )

print(f"Membaca {DATA_FILE} ...")
df_raw = pd.read_csv(DATA_FILE)
print(f"Total piksel mentah: {len(df_raw):,}")

Membaca D:\tugas psd\website-jupyterbook\UTS\data\piksel_s2.csv ...


Total piksel mentah: 613,282


### 1. Agregasi Tingkat Poligon (*Centroid Aggregation*)
Menghilangkan nilai infinite/NaN jika ada, kemudian menghitung nilai mean fitur spektral per `poligon_id`.

In [2]:
FITUR_SPEKTRAL = ['B02', 'B03', 'B04', 'B08', 'B11', 'NDVI', 'NDWI', 'MNDWI', 'NDBI']

# Bersihkan nilai invalid
df_clean = df_raw.replace([np.inf, -np.inf], np.nan).dropna(subset=FITUR_SPEKTRAL)

# Agregasi per poligon
df_poligon = df_clean.groupby('poligon_id').agg({
    'poligon_no': 'first',
    'sampel_id': 'first',
    'tile_id': 'first',
    'label': 'first',
    'label_teks': 'first',
    'lon': 'mean',
    'lat': 'mean',
    **{f: 'mean' for f in FITUR_SPEKTRAL}
}).reset_index()

print(f"Hasil agregasi poligon: {df_poligon.shape[0]} poligon unik x {df_poligon.shape[1]} kolom")
print("\nDistribusi Poligon per Kelas Tutupan Lahan:")
print(df_poligon['label_teks'].value_counts())
display(df_poligon.head())

Hasil agregasi poligon: 357 poligon unik x 17 kolom

Distribusi Poligon per Kelas Tutupan Lahan:
label_teks
Bangunan    60
Danau       60
Hutan       60
Mangrove    60
Laut        59
Sawah       58
Name: count, dtype: int64


,poligon_id,poligon_no,sampel_id,tile_id,label,label_teks,lon,lat,B02,B03,B04,B08,B11,NDVI,NDWI,MNDWI,NDBI
0,Bangunan_0000,0,58,x556_y-38,2,Bangunan,111.208307,-7.457741,709.684615,882.915385,917.838462,2455.284615,2480.357692,0.452892,-0.468145,-0.476165,0.007976
1,Bangunan_0001,1,59,x556_y-38,2,Bangunan,111.203330,-7.455503,683.794340,860.856604,962.190566,2475.658491,2655.532075,0.437712,-0.479927,-0.514773,0.039699
2,Bangunan_0002,2,60,x556_y-38,2,Bangunan,111.208395,-7.452758,713.092979,895.746679,949.420304,2549.561670,2566.597723,0.454925,-0.476062,-0.488206,0.008582
3,Bangunan_0003,3,61,x556_y-38,2,Bangunan,111.205926,-7.462115,820.887010,1000.555726,1093.528824,2421.211376,2585.983090,0.382561,-0.417816,-0.451327,0.036246
4,Bangunan_0004,4,62,x556_y-38,2,Bangunan,111.198358,-7.479653,575.022034,740.664972,789.594915,2492.803390,2345.640678,0.512376,-0.534572,-0.523829,-0.022646


### 2. Pembagian Data Latih dan Data Uji (Stratified Split 80:20)
Data poligon dibagi dengan `test_size=0.2`, `random_state=42`, dan `stratify=df_poligon['label']`.

In [3]:
# Stratified Split 80:20
train_idx, test_idx = train_test_split(
    df_poligon.index,
    test_size=0.20,
    random_state=42,
    stratify=df_poligon['label']
)

df_poligon['split'] = 'Train'
df_poligon.loc[test_idx, 'split'] = 'Test'

split_summary = pd.crosstab(df_poligon['label_teks'], df_poligon['split'], margins=True)
print("=== TABEL DISTRIBUSI SPLIT TRAIN-TEST ===")
display(split_summary)

n_train = (df_poligon['split'] == 'Train').sum()
n_test = (df_poligon['split'] == 'Test').sum()
print(f"Total Poligon Latih (Train): {n_train} poligon ({n_train/len(df_poligon)*100:.1f}%)")
print(f"Total Poligon Uji (Test)   : {n_test} poligon ({n_test/len(df_poligon)*100:.1f}%)")

=== TABEL DISTRIBUSI SPLIT TRAIN-TEST ===


split,Test,Train,All
label_teks,,,
Bangunan,12,48,60
Danau,12,48,60
Hutan,12,48,60
Laut,12,47,59
Mangrove,12,48,60
Sawah,12,46,58
All,72,285,357


Total Poligon Latih (Train): 285 poligon (79.8%)
Total Poligon Uji (Test)   : 72 poligon (20.2%)


### 3. Ekspor Data Poligon Terpisah ke Direktori Output
Menyimpan dataset poligon lengkap dengan kolom `split` ('Train' / 'Test') ke `output/df_poligon_split.csv` sebagai jembatan data bagi notebook selanjutnya.

In [4]:
berkas_split = OUT_DIR / "df_poligon_split.csv"
df_poligon.to_csv(berkas_split, index=False)

print(f"Dataset poligon dengan label split tersimpan di: {berkas_split.resolve()}")
print(f"Ukuran berkas: {berkas_split.stat().st_size:,} bytes")
print("=== DAFTAR BERKAS KELUARAN NOTEBOOK 03 ===")
print(f"- {berkas_split.name}")

Dataset poligon dengan label split tersimpan di: D:\tugas psd\website-jupyterbook\UTS\output\df_poligon_split.csv
Ukuran berkas: 82,310 bytes
=== DAFTAR BERKAS KELUARAN NOTEBOOK 03 ===
- df_poligon_split.csv
